# L13 · 特殊矩阵 — 正交矩阵保长度、对称矩阵 = 镜子、正定的判定

**目标**：认识单位阵、转置、逆矩阵；理解**正交矩阵**（保长度、易求逆）。

**为什么对 Aurora 重要**：`dft_matrix(n)` 返回酉矩阵，其逆等于共轭转置——这是 `ifft(fft(x)) == x` 成立的代数原因；加窗操作等价于对角矩阵 `diag(window)` 左乘信号向量。

## 本课剧情：给矩阵做体检

逆矩阵能还原变换，正交矩阵在还原的同时保持向量长度不变。这节课逐一验证 `A @ A_inv ≈ I` 和 `Q.T @ Q ≈ I`，最后写一个函数判断任意矩阵是否正交。

## 1. 单位阵、转置、逆

转置 `A.T` 把行换成列；逆矩阵 `A⁻¹` 满足 `A @ A⁻¹ = I`，把变换「撤销」。一般矩阵求逆需要 O(n³) 的 Gaussian 消元，但**正交矩阵**有捷径：`Q⁻¹ = Qᵀ`，转置即逆，既快又数值稳定。

**对称矩阵**（满足 `A = Aᵀ`）另有强性质：特征值全为实数，特征向量两两正交——协方差矩阵必须对称，这是 PCA 的代数基础。正交矩阵的复数推广是**酉矩阵**，`dft_matrix(n)` 属于此类，因此 `ifft(fft(x)) == x` 成立。

## 符号入口：先看形状，再看运算

向量是 `(n,)`，矩阵是 `(m, n)`，矩阵乘向量把 `(n,)` 变成 `(m,)`。遇到矩阵运算先确认 shape，再看数值。

In [ ]:
import numpy as np
I = np.eye(3); print('单位阵 I=\n', I)
A = np.array([[2.0,1.0],[1.0,3.0]])
print('转置 A.T=\n', A.T)
print('逆  A^-1=\n', np.linalg.inv(A))
print('A @ A^-1 ≈ I:\n', np.round(A @ np.linalg.inv(A), 6))

## 动手观察：矩阵的 shape 和逆

查看 `A`、`A.T`、`A_inv @ A` 的 shape 和数值，确认转置不改变元素总量，逆矩阵还原单位阵。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：同一矩阵作用于不同向量

遍历测试向量，观察普通矩阵如何改变 `np.linalg.norm`——与下面的正交矩阵对比。

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


## 2. 正交矩阵：列向量两两垂直且长度为 1 → `Qᵀ Q = I`，且保持向量长度不变

In [ ]:
theta = 1.0
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])  # 旋转矩阵 = 正交
x = np.array([3.0, 4.0])
print('QᵀQ =\n', np.round(Q.T @ Q, 6))
print('|x| =', np.linalg.norm(x), ' |Qx| =', np.linalg.norm(Q @ x), '(应相等)')

## 3. ✏️ 你的任务：判断一个矩阵是否正交

**推理路线**：
1. 正交矩阵定义：列向量两两正交且单位长度，等价于 `QᵀQ = I`
2. 验证只需一行：计算 `Q.T @ Q`，与 `np.eye(n)` 用 `np.allclose` 比较
3. 必须用 `np.allclose` 而非 `==`：`cos(θ)` 和 `sin(θ)` 有浮点误差，`1.0000000000000002 != 1.0`

**参考输入输出**：`Q=[[0,-1],[1,0]]`（90° 旋转）→ `True`；`M=[[1,2],[3,4]]` → `False`

<details><summary>点击查看参考实现</summary>

```python
def is_orthogonal(Q):
    n = len(Q)
    return bool(np.allclose(Q.T @ Q, np.eye(n)))
```

</details>

`Q.T @ Q` 的每个元素是两列向量之间的点积：对角线元素是列向量与自身的内积（应为 1，即单位长度），非对角线元素是不同列之间的内积（应为 0，即互相垂直）。整个矩阵约等于单位阵，就说明所有列两两正交且单位长度。

In [ ]:
def is_orthogonal(Q):
    # ✏️ TODO: 返回 True/False
    return None  # ← 改这里


In [ ]:
Q = np.array([[0.0,-1.0],[1.0,0.0]])   # 90° 旋转, 正交
M = np.array([[1.0, 2.0],[3.0, 4.0]])  # 不正交
print('Q 正交?', is_orthogonal(Q), '(应 True)')
print('M 正交?', is_orthogonal(M), '(应 False)')
assert is_orthogonal(Q) and not is_orthogonal(M)
print('\n✅ 通过：你能识别正交变换了。')

**🔗 Aurora 连接**：因为 FFT/DCT 矩阵正交，逆变换=转置(共轭转置)，所以 `ifft(fft(x)) == x`。这就是 Week 2 你会验证的事。

## 🎨 图示：正交矩阵(旋转)保持长度不变

In [ ]:
import numpy as np
from laviz import style, arrows2d
style()
th=0.9; R=np.array([[np.cos(th),-np.sin(th)],[np.sin(th),np.cos(th)]])
v=np.array([3.,1]); arrows2d([v, R@v], ['v','Qv (旋转,等长)'],
         title='正交矩阵 = 旋转/反射');

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：旋转角与对称矩阵构造

**实验 1**：把 cell 11 的旋转角 `theta` 改成 `np.pi/4`、`np.pi`、`0.0`，每次运行 `is_orthogonal`——正交性与角度无关，只要是旋转矩阵结果始终为 `True`。再换成 `np.diag([2, 3])`，确认缩放矩阵不是正交矩阵。

**实验 2**：生成随机矩阵 `R = np.random.randn(3, 3)`，计算 `S = R + R.T`，验证 `np.allclose(S, S.T)` 始终为 `True`。这是从任意矩阵构造对称矩阵的标准方法——对称矩阵的特征向量矩阵正是正交矩阵，两者在特征值分解中相遇。

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 本课收束

现在可以用 `is_orthogonal(Q)` 检验任意方阵，并用 `Q.T` 直接代替 `np.linalg.inv(Q)` 求逆。Aurora 的 `dft_matrix(n)` 是酉矩阵，Week 2 验证 `ifft(fft(x)) == x` 时直接依赖这条性质。下一节进入特征值分解，正交矩阵将作为特征向量矩阵再次出现。

In [ ]:
# 小检查：先看 shape，再做运算
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])
A = np.array([[1.0, 2.0], [0.0, 1.0]])

print('a.shape =', a.shape)
print('b.shape =', b.shape)
print('A.shape =', A.shape)
print('a + b =', a + b)
print('A @ a =', A @ a)
print('a dot b =', float(a @ b))
